# List of Lagrangians


## Setup


### Imports


In [31]:

import re
import sys
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import Expression, S

from feynpy import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    DC,
    Field,
    FieldTransformation,
    FS,
    Gamma,
    Gamma5,
    GaugeFixing,
    GaugeGroup,
    GaugeRepresentation,
    GhostField,
    GhostLagrangian,
    Model,
    Parameter,
    PartialD,
    flavor_index,
)
from feynpy.lagrangian import ComplexScalarKineticTerm
from lagrangian.operator_action import brst_transformation, gauge_variation
from symbolic.spenso_structures import (
    gauge_generator,
    lorentz_levi_civita,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")



def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = sum(source_terms[1:], source_terms[0]) if len(source_terms) > 1 else source_terms[0]
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields ))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)


### Symbols


In [32]:
mu, nu, rho = S("mu"), S("nu"), S("rho")
eQED, gS, g2, xiQCD = S("eQED"), S("gS"), S("g2"), S("xiQCD")
qPhi, qPsi, qQ, YL = S("qPhi"), S("qPsi"), S("qQ"), S("YL")
lam, y, g4, gGamma = S("lam"), S("y"), S("g4"), S("gGamma")
lam4, g, lamC, gD2, gijk = S("lam4"), S("g"), S("lamC"), S("gD2"), S("gijk")
yF, gV, gJJ, g_psi4, g1, g2f = S("yF"), S("gV"), S("gJJ"), S("g_psi4"), S("g1"), S("g2f")
i, j, k = S("i"), S("j"), S("k")
YH = S("YH")
xiW = S("xiW")


### Fields


In [33]:

Phi = Field(
    "Phi",
    spin=0,
    self_conjugate=False,
    symbol=S("phi"),
    conjugate_symbol=S("phidag"),
    quantum_numbers={"Q": qPhi},
)
Chi = Field(
    "Chi",
    spin=0,
    self_conjugate=False,
    symbol=S("chi"),
    conjugate_symbol=S("chidag"),
)
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": qPsi},
)
Xi = Field(
    "Xi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("xi"),
    conjugate_symbol=S("xibar"),
    indices=(SPINOR_INDEX,),
)
Quark = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("q"),
    conjugate_symbol=S("qbar"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Q": qQ},
)
Photon = Field("A", spin=1, self_conjugate=True, symbol=S("A0"), indices=(LORENTZ_INDEX,))
Gluon = Field("G", spin=1, self_conjugate=True, symbol=S("G0"), indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX))
GhostG = GhostField(
    "ghG",
    ghost_of=Gluon,
    self_conjugate=False,
    symbol=S("ghG0"),
    conjugate_symbol=S("ghGbar0"),
    indices=(COLOR_ADJ_INDEX,),
    quantum_numbers={"GhostNumber": 1},
)
W = Field("W", spin=1, self_conjugate=True, symbol=S("W0"), indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX))
GhostW = GhostField(
    "ghW",
    ghost_of=W,
    self_conjugate=False,
    symbol=S("ghW0"),
    conjugate_symbol=S("ghWbar0"),
    indices=(WEAK_ADJ_INDEX,),
    quantum_numbers={"GhostNumber": 1},
)
L = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L0"),
    conjugate_symbol=S("Lbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": YL},
)

B = Field(
    "B",
    spin=1,
    self_conjugate=True,
    symbol=S("B0"),
    indices=(LORENTZ_INDEX,),
)
HY = Field(
    "HY",
    spin=0,
    self_conjugate=False,
    symbol=S("HY0"),
    conjugate_symbol=S("HYdag0"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": YH},
)


# Compact scalar-only species for the short model-layer examples below.
PhiR = Field("PhiR", spin=0, self_conjugate=True, symbol=S("phi0"))
ChiR = Field("ChiR", spin=0, self_conjugate=True, symbol=S("chi0"))
PhiC = Field("PhiC", spin=0, self_conjugate=False, symbol=S("phiC0"), conjugate_symbol=S("phiCdag0"))
Phi_i = Field("phi_i", spin=0, self_conjugate=True, symbol=S("phi_i0"))
Phi_j = Field("phi_j", spin=0, self_conjugate=True, symbol=S("phi_j0"))
Phi_k = Field("phi_k", spin=0, self_conjugate=True, symbol=S("phi_k0"))

(Phi, Chi, Psi, Xi, Quark, Photon, Gluon, GhostG, W, GhostW, HY, L)


(Field(name='Phi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=phi, conjugate_symbol=phidag, mass=None, quantum_numbers=mappingproxy({'Q': qPhi}), ghost_of=None, goldstone_of=None, flavor_index=None, class_members=()),
 Field(name='Chi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=chi, conjugate_symbol=chidag, mass=None, quantum_numbers=mappingproxy({}), ghost_of=None, goldstone_of=None, flavor_index=None, class_members=()),
 Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', dimension=4, is_flavor=False, prefix='i'),), kind='fermion', statistics='fermion', symbol=psi, conjugate_symbol=psibar, mass=None, quantum_numbers=mappingproxy({'Q': qPsi}), ghost_of=None, goldstone_of=None, flavor_index=None, class_members=()),
 Field(name='Xi', spin=Fraction(1, 2), self_conjugate=False, ind

### Gauge Representations


In [34]:
COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fund",
)
WEAK_DOUBLET_REP = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_gauge_generator,
    name="doublet",
)

(COLOR_FUND_REP, WEAK_DOUBLET_REP)


(GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', dimension=3, is_flavor=False, prefix='c'), generator_builder=<function gauge_generator at 0x10acf7110>, name='fund', slot=None, slot_policy='unique'),
 GaugeRepresentation(index=IndexType(name='WeakFund', representation=Representation { rep: Dualizable(3), dim: Concrete(2) }, kind='weak_fund', dimension=2, is_flavor=False, prefix='w'), generator_builder=<function weak_gauge_generator at 0x10acf7270>, name='doublet', slot=None, slot_policy='unique'))

### Gauge Groups


In [35]:
U1QED = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=eQED,
    gauge_boson=Photon,
    charge="Q",
)
U1Y = GaugeGroup(
    name="U1Y",
    abelian=True,
    coupling=g1,
    gauge_boson=B,
    charge="Y",
)
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=Gluon,
    ghost_field=GhostG,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W,
    ghost_field=GhostW,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

(U1QED, SU3C, SU2L)


(GaugeGroup(name='U1QED', abelian=True, coupling=eQED, gauge_boson=Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=4, is_flavor=False, prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers=mappingproxy({}), ghost_of=None, goldstone_of=None, flavor_index=None, class_members=()), ghost_field=None, structure_constant=None, representations=(), charge='Q'),
 GaugeGroup(name='SU3C', abelian=False, coupling=gS, gauge_boson=Field(name='G', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=4, is_flavor=False, prefix='mu'), IndexType(name='ColorAdj', representation=Representation { rep: SelfDual(2), dim: Concrete(8) }, kind='color_adj', dimension=8, is_flavor=False, prefix='a')), kind='vector', st

## Scalar Examples


In [36]:
phi4 = Model(lam4 * PhiR * PhiR * PhiR * PhiR)
show_model(phi4)



Lagrangian
lam4 * PhiR * PhiR * PhiR * PhiR

Feynman Rules
1 vertex signature(s)

Vertex: ('PhiR', 'PhiR', 'PhiR', 'PhiR')
Rule: 24𝑖*lam4



In [37]:

derivative_phi4 = Model(
    gD2 * PartialD(PhiR, mu) * PartialD(PhiR, mu) * PhiR * PhiR,
)
show_model(derivative_phi4)

Lagrangian
gD2 * PartialD(PhiR, mu) * PartialD(PhiR, mu) * PhiR * PhiR

Feynman Rules
1 vertex signature(s)

Vertex: ('PhiR', 'PhiR', 'PhiR', 'PhiR')
Rule: -4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)-4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q3,mu1_int)-4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q4,mu1_int)-4𝑖*gD2*pcomp(q2,mu1_int)*pcomp(q3,mu1_int)-4𝑖*gD2*pcomp(q2,mu1_int)*pcomp(q4,mu1_int)-4𝑖*gD2*pcomp(q3,mu1_int)*pcomp(q4,mu1_int)



## Fermion Examples


In [38]:

qed_fermion_model = Model(
    I * Psi.bar * Gamma(mu) * DC(Psi, mu),
    gauge_groups=(U1QED,),
)
show_model(qed_fermion_model)

qcd_fermion_model = Model(
    I * Quark.bar * Gamma(mu) * DC(Quark, mu),
    gauge_groups=(SU3C,),
)
show_model(qcd_fermion_model)

photon_kinetic_model = Model(
    -(Expression.num(1) / Expression.num(4)) * FS(U1QED, mu, nu) * FS(U1QED, mu, nu),
)
show_model(photon_kinetic_model)

gluon_kinetic_model = Model(
    -(Expression.num(1) / Expression.num(4)) * FS(SU3C, mu, nu, S("aC")) * FS(SU3C, mu, nu, S("aC")),
)
show_model(gluon_kinetic_model)

Lagrangian
1𝑖 * Psi.bar * Gamma(mu) * DC(Psi, mu)

Feynman Rules
2 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'A')
Rule: 1𝑖*eQED*qPsi*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Vertex: ('Psi.bar', 'Psi')
Rule: 1𝑖*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)

Lagrangian
1𝑖 * q.bar * Gamma(mu) * DC(q, mu)

Feynman Rules
2 vertex signature(s)

Vertex: ('q.bar', 'q', 'G')
Rule: 1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))

Vertex: ('q.bar', 'q')
Rule: 1𝑖*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)

Lagrangian
-1/4 * FS(U1QED, mu, nu) * FS(U1QED, mu, nu)

Feynman Rules
1 vertex signature(s)

Vertex: ('A', 'A')
Rule: -1𝑖*pcomp(q1,mu2)*pcomp(q2,mu1)+1𝑖*g(mink(4, mu1),mink(4, mu2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)

Lagrangian
-1/4 * FS(SU3C, mu, nu, aC) * FS(SU3C, mu, nu, aC)

Feynman Rules
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),

In [39]:

xiQED = S("xiQED")

qed_gf_helper = Model(
    GaugeFixing(U1QED, xi=xiQED),
)
qed_gf_manual = Model(
    (
        -(Expression.num(1) / (Expression.num(2) * xiQED))
        * PartialD(Photon(S("mu_qed_gf")), S("mu_qed_gf"))
        * PartialD(Photon(S("nu_qed_gf")), S("nu_qed_gf"))
    ),
)

show(
    "QED Gauge-Fixing Helper Rule",
    qed_gf_helper.lagrangian().feynman_rule(Photon, Photon ),
)
show(
    "QED Gauge-Fixing Manual Rule",
    qed_gf_manual.lagrangian().feynman_rule(Photon, Photon ),
)

qcd_gf_helper = Model(
    GaugeFixing(SU3C, xi=xiQCD),
)
qcd_gf_manual = Model(
    (
        -(Expression.num(1) / (Expression.num(2) * xiQCD))
        * PartialD(Gluon(S("mu_qcd_gf"), S("a_qcd_gf")), S("mu_qcd_gf"))
        * PartialD(Gluon(S("nu_qcd_gf"), S("a_qcd_gf")), S("nu_qcd_gf"))
    ),
)

show(
    "QCD Gauge-Fixing Helper Rule",
    qcd_gf_helper.lagrangian().feynman_rule(Gluon, Gluon ),
)
show(
    "QCD Gauge-Fixing Manual Rule",
    qcd_gf_manual.lagrangian().feynman_rule(Gluon, Gluon ),
)

qcd_gf_ghost_direct = Model(
    GaugeFixing(SU3C, xi=xiQCD) - GhostG.bar * PartialD(DC(GhostG, mu), mu),
)
show(
    "QCD Gauge-Fixing + Ghost Rules",
    qcd_gf_ghost_direct.lagrangian().feynman_rule(include_delta=False),
)


QED Gauge-Fixing Helper Rule
1𝑖*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQED

QED Gauge-Fixing Manual Rule
1𝑖*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQED

QCD Gauge-Fixing Helper Rule
1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQCD

QCD Gauge-Fixing Manual Rule
1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQCD

QCD Gauge-Fixing + Ghost Rules
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQCD

Vertex: ('ghG.bar', 'ghG')
Rule: 1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q2,mu1_int)^2

Vertex: ('ghG.bar', 'ghG', 'G')
Rule: gS*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q2,mu3)+gS*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q3,mu3)



## Features

### Flavor Expansion

In [40]:
Generation = flavor_index("Generation", 3, prefix="f")
f, h = S("f"), S("h")

Leptons = Field(
    "Leptons",
    spin=Fraction(1, 2),
    self_conjugate=False,
    indices=(Generation, SPINOR_INDEX),
    flavor_index=Generation,
    class_members=("e", "mu", "tau"),
)
electron, muon, tau = Leptons.class_members
Yukawa = Parameter("Yukawa", symbol=S("Y"), indices=(Generation, Generation))

flavor_model = Model(
    Yukawa(f, h) * Leptons.bar(f) * Leptons(h) * PhiR,
    parameters=(Yukawa,),
)

show(
    "Compact Flavor Rule",
    flavor_model.feynman_rule(),
)
show(
    "Expanded Flavor Rules",
    flavor_model.feynman_rule(flavor_expand=True),
)
show(
    "Selected Mixing Rule: e.bar, mu, PhiR",
    flavor_model.feynman_rule(
        electron.bar,
        muon,
        PhiR,
        flavor_expand=True,
    ),
)


Compact Flavor Rule
1 vertex signature(s)

Vertex: ('Leptons.bar', 'Leptons', 'PhiR')
Rule: 1𝑖*g(bis(4, i1),bis(4, i2))*Y(f1,f2)

Expanded Flavor Rules
9 vertex signature(s)

Vertex: ('e.bar', 'e', 'PhiR')
Rule: 1𝑖*g(bis(4, i1),bis(4, i2))*Y(1,1)

Vertex: ('e.bar', 'mu', 'PhiR')
Rule: 1𝑖*g(bis(4, i1),bis(4, i2))*Y(1,2)

Vertex: ('e.bar', 'tau', 'PhiR')
Rule: 1𝑖*g(bis(4, i1),bis(4, i2))*Y(1,3)

Vertex: ('mu.bar', 'e', 'PhiR')
Rule: 1𝑖*g(bis(4, i1),bis(4, i2))*Y(2,1)

Vertex: ('mu.bar', 'mu', 'PhiR')
Rule: 1𝑖*g(bis(4, i1),bis(4, i2))*Y(2,2)

Vertex: ('mu.bar', 'tau', 'PhiR')
Rule: 1𝑖*g(bis(4, i1),bis(4, i2))*Y(2,3)

Vertex: ('tau.bar', 'e', 'PhiR')
Rule: 1𝑖*g(bis(4, i1),bis(4, i2))*Y(3,1)

Vertex: ('tau.bar', 'mu', 'PhiR')
Rule: 1𝑖*g(bis(4, i1),bis(4, i2))*Y(3,2)

Vertex: ('tau.bar', 'tau', 'PhiR')
Rule: 1𝑖*g(bis(4, i1),bis(4, i2))*Y(3,3)

Selected Mixing Rule: e.bar, mu, PhiR
1𝑖*g(bis(4, i1),bis(4, i2))*Y(1,2)



### Symbolica Export

In [41]:
symbolica_expression = flavor_model.to_symbolica(flavor_expand=True)
show("Symbolica Expression", symbolica_expression)
type(symbolica_expression)


Symbolica Expression
phi0*mu(i_decl_1)*ebar(i_decl_1)*Y(1,2)-phi0*mu(i_decl_1)*mubar(i_decl_1)*Y(2,2)-phi0*mu(i_decl_1)*taubar(i_decl_1)*Y(3,2)-phi0*ebar(i_decl_1)*e(i_decl_1)*Y(1,1)+phi0*ebar(i_decl_1)*tau(i_decl_1)*Y(1,3)-phi0*e(i_decl_1)*mubar(i_decl_1)*Y(2,1)-phi0*e(i_decl_1)*taubar(i_decl_1)*Y(3,1)+phi0*mubar(i_decl_1)*tau(i_decl_1)*Y(2,3)-phi0*taubar(i_decl_1)*tau(i_decl_1)*Y(3,3)



symbolica.core.Expression

### Rule Output and Simplification

In [42]:
raw_qcd_rule = qcd_fermion_model.feynman_rule(
    Quark.bar,
    Quark,
    Gluon,
    simplify=False,
    strip_externals=False,
    include_delta=True,
)
simple_qcd_rule = qcd_fermion_model.feynman_rule(
    Quark.bar,
    Quark,
    Gluon,
    simplify=True,
    simplify_gamma=True,
    strip_externals=True,
    include_delta=False,
)

show("Raw Rule", raw_qcd_rule)
show("Simplified Rule", simple_qcd_rule)


Raw Rule
1𝑖*gS*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))*delta(q,q)*delta(qbar,qbar)*delta(G0,G0)*U(G0,q3)*Delta(q1+q2+q3)

Simplified Rule
1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))



### Higher-Dimensional Field-Strength Operators

In [43]:
theta, c3 = S("theta"), S("c3")
sigma = S("sigma")
a, b, c = S("a"), S("b"), S("c")

cp_odd_model = Model(
    theta
    * lorentz_levi_civita(mu, nu, rho, sigma)
    * FS(U1QED, mu, nu)
    * FS(U1QED, rho, sigma),
)

triple_gluon_model = Model(
    c3
    * structure_constant(a, b, c)
    * FS(SU3C, mu, nu, a)
    * FS(SU3C, nu, rho, b)
    * FS(SU3C, rho, mu, c),
)

show("Epsilon F F", cp_odd_model.lagrangian_decl)
show("Epsilon F F Signatures", cp_odd_model.vertex_signatures())
show("f F F F", triple_gluon_model.lagrangian_decl)
show("f F F F Signatures", triple_gluon_model.vertex_signatures())


Epsilon F F
theta*lor_levi_civita(mink(4, mu),mink(4, nu),mink(4, rho),mink(4, sigma)) * FS(U1QED, mu, nu) * FS(U1QED, rho, sigma)

Epsilon F F Signatures
(VertexSignature(fields=(Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=4, is_flavor=False, prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers=mappingproxy({}), ghost_of=None, goldstone_of=None, flavor_index=None, class_members=()), Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=4, is_flavor=False, prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers=mappingproxy({}), ghost_of=None, goldstone_of=None, flavor_index=None, class_members=())), names=('A', 'A'

### Gauge and BRST Transformations

In [44]:
a_transform = S("a_transform")
Baux = Field(
    "Baux",
    spin=0,
    self_conjugate=True,
    indices=(COLOR_ADJ_INDEX,),
)

gluon_field_model = Model(
    Gluon(mu, a_transform),
    gauge_groups=(SU3C,),
    fields=(Gluon, GhostG, Baux),
)

delta_su3 = gauge_variation(group=SU3C, parameter="alpha")
brst_su3 = brst_transformation(
    group=SU3C,
    ghost=GhostG,
    auxiliary=Baux,
)

gauge_transformed_gluon = gluon_field_model.apply_operator(delta_su3)
brst_transformed_gluon = gluon_field_model.apply_operator(brst_su3)

show("Gauge Transformation of G", gauge_transformed_gluon.to_symbolica())
show("BRST Transformation of G", brst_transformed_gluon.to_symbolica())


Gauge Transformation of G
-gS*f(coad(8, a_transform),coad(8, a_delta_SU3C_1),coad(8, a_delta_SU3C_2))*G0(mu,a_delta_SU3C_2)*alpha(a_delta_SU3C_1)+PartialD(alpha(a_transform),mu)

BRST Transformation of G
gS*f(coad(8, a_transform),coad(8, a_brst_SU3C_1),coad(8, a_brst_SU3C_2))*G0(mu,a_brst_SU3C_1)*ghG0(a_brst_SU3C_2)+PartialD(ghG0(a_transform),mu)



### Field Transformations and Mixing

In [45]:
EtaR = Field("EtaR", spin=0, self_conjugate=True, symbol=S("eta0"))
cMix, sMix, mMix = S("cMix"), S("sMix"), S("mMix")

mixing_model = Model(mMix * PhiR * PhiR)
mixed_model = mixing_model.transform_fields(
    FieldTransformation(PhiR, cMix * ChiR + sMix * EtaR),
    repeat=False,
)

show("Before Mixing", mixing_model.lagrangian_decl)
show("After Mixing", mixed_model.to_symbolica())
show("Mixed Rules", mixed_model.feynman_rule(include_delta=False))


Before Mixing
mMix * PhiR * PhiR

After Mixing
2*chi0*eta0*cMix*sMix*mMix+chi0^2*cMix^2*mMix+eta0^2*sMix^2*mMix

Mixed Rules
3 vertex signature(s)

Vertex: ('ChiR', 'ChiR')
Rule: 2𝑖*cMix^2*mMix

Vertex: ('ChiR', 'EtaR')
Rule: 2𝑖*cMix*sMix*mMix

Vertex: ('EtaR', 'EtaR')
Rule: 2𝑖*sMix^2*mMix

